# 02 — Download Scripts

Runs the scrapers in `download_scripts/` that populate `data/` before the
downstream notebooks (`03_targets`, `04_controls`, `05_alphas`, ...) can run.

Pipeline:
1. **BSE CG downloader** — scrapes Corporate Governance XBRL filings for every company in `data/raw/top_500_companies.xlsx` → `data/raw/governance_reports/`
2. **Finalize dataset** — retries pending failures, deletes invalid files, zips `data/raw/governance_reports/` → `data/raw/governance_reports.zip`
3. **Build filing dates DB** — walks `data/raw/governance_reports/` on disk → `data/processed/filing_dates_db.csv` (consumed by `03_targets` and `05_alphas`)
4. **NSE annual report downloader** — independent branch, downloads FY23-25 annual report PDFs → `data/raw/annual_reports/`

Steps 1, 2 and 4 are long-running (real browser / network I/O) — run cells individually rather than "Run All" the first time.

Each script skips its own work when its output already exists, so re-running this notebook is safe: step 1 and 2 no-op once `governance_reports.zip` exists, step 3 no-ops once `filing_dates_db.csv` exists, and step 4 skips any company/FY already recorded as downloaded in the manifest. Delete the relevant output to force a re-run.

Requires `matched_companies.xlsx` (from `00_cleaning_prowess`) and `industry_map.xlsx` (from `01_industries_map`) to already exist in `data/`.

In [ ]:
import subprocess
import sys
from pathlib import Path

BASE    = Path.cwd()
SCRIPTS = BASE / "download_scripts"

def run_script(script_name: str, *args: str) -> None:
    """Run a download_scripts/*.py file as a subprocess, streaming its output."""
    cmd = [sys.executable, str(SCRIPTS / script_name), *args]
    print("→", " ".join(cmd))
    subprocess.run(cmd, cwd=BASE, check=True)

def run_module(module_name: str, *args: str) -> None:
    """Run a download_scripts module (needed for finalize_dataset.py, which does
    `from download_scripts.bse_cg_downloader import ...` and only resolves when
    invoked as `python -m download_scripts.<module>` from the repo root)."""
    cmd = [sys.executable, "-m", f"download_scripts.{module_name}", *args]
    print("→", " ".join(cmd))
    subprocess.run(cmd, cwd=BASE, check=True)


## Step 1 — BSE Corporate Governance downloader

Opens a visible (non-headless) Chromium browser via Playwright — required to bypass
BSE's bot detection. `--batch` slices the company list into chunks of `--batch-size`
(default 100) if you'd rather not run all ~500 companies in one sitting.

Skips entirely if `data/raw/governance_reports.zip` already exists (dataset already finalized).

In [ ]:
run_script("bse_cg_downloader.py", "--xlsx", "data/raw/top_500_companies.xlsx")

# Batched alternative, e.g. first 100 companies only:
# run_script("bse_cg_downloader.py", "--xlsx", "data/raw/top_500_companies.xlsx", "--batch", "1")


## Step 2 — Finalize dataset

Retries any quarters left in `data/logs/cg_failed_downloads.csv`, deletes files that
slipped through without a valid CG filing, then zips `data/raw/governance_reports/` into
`data/raw/governance_reports.zip`.

Skips entirely if `data/raw/governance_reports.zip` already exists — delete it to re-run.

In [ ]:
run_module("finalize_dataset")


## Step 3 — Build filing dates database

Walks the files actually on disk in `data/raw/governance_reports/` and cross-references
`data/logs/cg_download_log.csv`, `data/processed/industry_map.xlsx` and `data/processed/matched_companies.xlsx`
to produce `data/processed/filing_dates_db.csv`.

Skips entirely if `data/processed/filing_dates_db.csv` already exists — delete it to rebuild.

In [ ]:
run_script("build_filing_dates_db.py")


## Step 4 — NSE annual report downloader

Independent of the CG pipeline above — downloads FY23/FY24/FY25 annual report PDFs
for companies in `data/processed/matched_companies.xlsx` from NSE. `--company` filters to a
single company by name or BSE code for a quick test run.

Per-company/FY reports already recorded as downloaded in `data/logs/ar_manifest.json`
are skipped; if every requested report is already downloaded, the whole run no-ops.

In [ ]:
run_script("nse_ar_downloader.py")

# Quick single-company test:
# run_script("nse_ar_downloader.py", "--company", "INFOSYS")
